In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from xgboost import XGBClassifier

from sklearn.model_selection import StratifiedKFold
import optuna
from sklearn.base import clone
from sklearn.metrics import average_precision_score

In [2]:
import optuna.visualization as vis
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook"

In [3]:
RANDOM_STATE = 42

In [4]:
df = pd.read_csv("../data/diabetes_binary_health_indicators_BRFSS2015.csv")
df.head()

,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


In [5]:
_target_cols = ['Diabetes_binary']
_binary_cols = ['HighBP', 'HighChol', 'CholCheck', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity',
                'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'DiffWalk', 'Sex']
_ordinal_cols = ['GenHlth', 'Education', 'Income', 'Age']
_gray_cols = ['MentHlth', 'PhysHlth']
_continuous_cols = ["BMI"]

In [6]:
x = df.drop(_target_cols, axis=1)
y = df[_target_cols].to_numpy().ravel()
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

In [7]:
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
POS_WEIGHT_RATIO = (y_train == 0).sum() / (y_train == 1).sum()
sampler_kwargs = dict(seed=RANDOM_STATE)

In [8]:
def _cv_score_with_pruning(trial, pipeline, x, y, cv, fit_kwargs_fn=None) -> float:
    fold_scores = []
    for fold_idx, (train_idx, val_idx) in enumerate(cv.split(x, y)):
        x_tr, x_val = x.iloc[train_idx], x.iloc[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        fold_pipeline = clone(pipeline)
        fit_kwargs = fit_kwargs_fn(x_val, y_val) if fit_kwargs_fn is not None else {}
        fold_pipeline.fit(x_tr, y_tr, **fit_kwargs)

        y_prob = fold_pipeline.predict_proba(x_val)[:, 1]
        fold_scores.append(average_precision_score(y_val, y_prob))

        trial.report(np.mean(fold_scores), step=fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(fold_scores))

In [9]:
xgb_model = XGBClassifier(
    device="cpu",
    random_state=RANDOM_STATE,
)


xgb_pipeline = Pipeline([
    ('xgb', xgb_model)
])


xgb_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('xgb', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",'cpu'


In [10]:
def xgb_parameter_space(trial: optuna.Trial) -> dict:
    params = {
        "xgb__n_estimators": 3000,
        "xgb__early_stopping_rounds": 50,
        "xgb__max_depth": trial.suggest_int("max_depth", 3, 10),
        "xgb__learning_rate": trial.suggest_float("learning_rate", 0.003, 0.3, log=True),
        "xgb__subsample": trial.suggest_float("subsample", 0.4, 1.0),
        "xgb__colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "xgb__colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.5, 1.0),
        "xgb__min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "xgb__gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "xgb__reg_alpha": trial.suggest_float("reg_alpha", 1e-9, 1000.0, log=True),
        "xgb__reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 1000.0, log=True),
        "xgb__scale_pos_weight": trial.suggest_float(
            "scale_pos_weight", POS_WEIGHT_RATIO * 0.5, POS_WEIGHT_RATIO * 1.6
        ),
        "xgb__objective": "binary:logistic",
        "xgb__eval_metric": "aucpr",
        "xgb__tree_method": "hist",
        "xgb__booster": "gbtree",
    }
    return params

In [11]:
def xgb_objective(trial, x, y):
    space = xgb_parameter_space(trial)
    space["xgb__n_jobs"] = 1
    pipeline = clone(xgb_pipeline).set_params(**space)

    def xgb_fit_kwargs(x_val, y_val):
        return {"xgb__eval_set": [(x_val, y_val)], "xgb__verbose": False}

    return _cv_score_with_pruning(trial, pipeline, x, y, CV, fit_kwargs_fn=xgb_fit_kwargs)

In [ ]:
xgb_study = optuna.create_study(
    direction="maximize", study_name="xgb_tuning",
    sampler=optuna.samplers.TPESampler(**sampler_kwargs),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2),
)
xgb_study.optimize(lambda t: xgb_objective(t, x_train, y_train), n_trials=100, n_jobs=-1)

[I 2026-07-18 14:17:39,130] A new study created in memory with name: xgb_tuning
[I 2026-07-18 14:18:08,129] Trial 6 finished with value: 0.4314231129002886 and parameters: {'max_depth': 8, 'learning_rate': 0.23756240253641034, 'subsample': 0.8822301552990374, 'colsample_bytree': 0.4966178174863898, 'colsample_bylevel': 0.8486292949895313, 'min_child_weight': 2, 'gamma': 1.426898390482854, 'reg_alpha': 0.4485321755408594, 'reg_lambda': 1.5223868391748618, 'scale_pos_weight': 8.539025730126394}. Best is trial 6 with value: 0.4314231129002886.
[I 2026-07-18 14:18:09,797] Trial 2 finished with value: 0.4353254966208978 and parameters: {'max_depth': 6, 'learning_rate': 0.19051139881154325, 'subsample': 0.41565939618115405, 'colsample_bytree': 0.6747624795714444, 'colsample_bylevel': 0.9572481763219897, 'min_child_weight': 10, 'gamma': 3.0916457298727824, 'reg_alpha': 0.23251578576213303, 'reg_lambda': 4.4656769618641464e-06, 'scale_pos_weight': 7.256826493466825}. Best is trial 2 with value

In [ ]:
print(xgb_study.best_value, xgb_study.best_params)

In [ ]:
def per_model_plots(study):
    vis.plot_optimization_history(study).show()
    vis.plot_param_importances(study).show()
    vis.plot_intermediate_values(study).show()
    vis.plot_slice(study).show()
    vis.plot_parallel_coordinate(study).show()
    vis.plot_contour(study).show()

In [ ]:
per_model_plots(xgb_study)

In [ ]:
def build_best_pipeline(base_pipeline, param_space_fn, best_params, extra_params=None):
    fixed_trial = optuna.trial.FixedTrial(best_params)
    params = param_space_fn(fixed_trial)
    if extra_params:
        params.update(extra_params)
    return clone(base_pipeline).set_params(**params)


xgb_best_pipeline = build_best_pipeline(xgb_pipeline, xgb_parameter_space, xgb_study.best_params)

x_train_fit, x_train_es, y_train_fit, y_train_es = train_test_split(
    x_train, y_train, test_size=0.1, stratify=y_train, random_state=RANDOM_STATE
)

xgb_best_pipeline.fit(x_train_fit, y_train_fit, xgb__eval_set=[(x_train_es, y_train_es)], xgb__verbose=False)

final_n_estimators = xgb_best_pipeline.named_steps["xgb"].best_iteration + 1
print("XGBoost trees actually used:", final_n_estimators)


In [ ]:
import json
from pathlib import Path
from datetime import datetime

RESULTS_PATH = Path("../model/tuning_results.json")
MODEL_NAME = "xgboost"


def load_results(path: Path) -> dict:
    if path.exists():
        with open(path, "r") as f:
            return json.load(f)
    return {}


def save_best_if_improved(path: Path, model_name: str, score: float, params: dict, metric: str) -> dict:
    results = load_results(path)
    previous = results.get(model_name)

    if previous is None or score > previous["best_score"]:
        results[model_name] = {
            "best_score": score,
            "metric": metric,
            "best_params": params,
            "updated_at": datetime.now().isoformat(),
        }
        with open(path, "w") as f:
            json.dump(results, f, indent=2)

        if previous is None:
            print(f"[{model_name}] No previous record found. Saved best score: {score:.5f}")
        else:
            print(
                f"[{model_name}] New best score {score:.5f} beat the previous best "
                f"{previous['best_score']:.5f} (saved {previous['updated_at']}). Updated {path}."
            )
    else:
        print(
            f"[{model_name}] Score {score:.5f} did NOT beat the current all-time best "
            f"{previous['best_score']:.5f} (saved {previous['updated_at']}). Keeping existing parameters."
        )

    return results

fixed_trial = optuna.trial.FixedTrial(xgb_study.best_params)
prefixed_params = xgb_parameter_space(fixed_trial)
xgb_best_params_clean = {k.replace("xgb__", "", 1): v for k, v in prefixed_params.items()}
xgb_best_params_clean["n_estimators"] = final_n_estimators
xgb_best_params_clean["early_stopping_rounds"] = None

results = save_best_if_improved(
    RESULTS_PATH,
    MODEL_NAME,
    score=xgb_study.best_value,
    params=xgb_best_params_clean,
    metric="average_precision (5-fold stratified CV)",
)
